In [ ]:
import pandas as pd

def clean_match_df(df):
    # Split score "4–2" into home_goals, away_goals
    df["home_goals"] = df["Score"].str.extract(r"(\d+)[–-](\d+)")[0].astype(int)
    df["away_goals"] = df["Score"].str.extract(r"(\d+)[–-](\d+)")[1].astype(int)

    # Rename xG columns
    df = df.rename(columns={
        "Home": "home_team",
        "Away": "away_team",
        "xG": "home_xg",
        "xG.1": "away_xg"
    })

    # Create label: win/draw/loss from home perspective
    df["result"] = df.apply(
        lambda row: 
            "W" if row.home_goals > row.away_goals
            else "L" if row.home_goals < row.away_goals
            else "D",
        axis=1
    )

    return df


In [ ]:
currmatches = pd.read_csv('../../data/premier_league_matches_2025_2026.csv')

In [ ]:
currmatches["Date"] = pd.to_datetime(currmatches["Date"])

In [ ]:
# Keep only matches that have actually been played (have a recorded score).
# Using Score instead of a "today" date cutoff keeps this notebook reproducible
# regardless of when it is re-run, since the raw scrape also includes future fixtures.
currmatches = currmatches[currmatches["Score"].notna()]

In [ ]:
currmatches

In [ ]:
clean_matches = clean_match_df(currmatches)
clean_matches

In [ ]:
def add_rolling_features(df, window=5):

    df = df.sort_values("Date")

    # Team-based dictionaries for rolling stats
    team_stats = {}

    home_stats = []
    away_stats = []

    for _, row in df.iterrows():
        home = row.home_team
        away = row.away_team

        # Initialize if needed
        for t in [home, away]:
            if t not in team_stats:
                team_stats[t] = {
                    "gf": [], "ga": [], "pts": []
                }

        # Compute home rolling
        home_form = team_stats[home]
        away_form = team_stats[away]

        home_stats.append({
            "home_avg_gf_5": pd.Series(home_form["gf"][-window:]).mean(),
            "home_avg_ga_5": pd.Series(home_form["ga"][-window:]).mean(),
            "home_avg_pts_5": pd.Series(home_form["pts"][-window:]).mean(),
            "home_form_score": sum(home_form["pts"][-window:])
        })

        # Compute away rolling
        away_stats.append({
            "away_avg_gf_5": pd.Series(away_form["gf"][-window:]).mean(),
            "away_avg_ga_5": pd.Series(away_form["ga"][-window:]).mean(),
            "away_avg_pts_5": pd.Series(away_form["pts"][-window:]).mean(),
            "away_form_score": sum(away_form["pts"][-window:])
        })

        # Update team stats after computing features
        home_pts = 3 if row.home_goals > row.away_goals else 1 if row.home_goals == row.away_goals else 0
        away_pts = 3 if row.away_goals > row.home_goals else 1 if row.home_goals == row.away_goals else 0

        home_form["gf"].append(row.home_goals)
        home_form["ga"].append(row.away_goals)
        home_form["pts"].append(home_pts)

        away_form["gf"].append(row.away_goals)
        away_form["ga"].append(row.home_goals)
        away_form["pts"].append(away_pts)

    home_df = pd.DataFrame(home_stats)
    away_df = pd.DataFrame(away_stats)

    return pd.concat([df.reset_index(drop=True), home_df, away_df], axis=1)


In [ ]:
currmatches = add_rolling_features(clean_matches, window=5)
currmatches

In [ ]:
def update_elo(df, base_elo=1500, k=20):

    elos = {}
    home_elos = []
    away_elos = []

    for _, row in df.iterrows():
        h = row.home_team
        a = row.away_team

        if h not in elos: elos[h] = base_elo
        if a not in elos: elos[a] = base_elo

        home_elos.append(elos[h])
        away_elos.append(elos[a])

        # Determine actual score
        if row.home_goals > row.away_goals:
            score_h, score_a = 1, 0
        elif row.home_goals < row.away_goals:
            score_h, score_a = 0, 1
        else:
            score_h, score_a = 0.5, 0.5

        # Expected score
        expected_h = 1 / (1 + 10 ** ((elos[a] - elos[h]) / 400))
        expected_a = 1 - expected_h

        # Update
        elos[h] += k * (score_h - expected_h)
        elos[a] += k * (score_a - expected_a)

    df["home_elo"] = home_elos
    df["away_elo"] = away_elos
    return df


In [ ]:
currmatches = update_elo(currmatches)
currmatches

In [ ]:
currmatches.drop(columns=['Score', 'Wk', 'Day', 'Time', 'Score', 'home_goals', 'away_goals', ], axis=1, inplace=True)

In [ ]:
rolling_cols = [
    "home_avg_gf_5", "home_avg_ga_5", "home_avg_pts_5", "home_form_score",
    "away_avg_gf_5", "away_avg_ga_5", "away_avg_pts_5", "away_form_score"
]
currmatches[rolling_cols] = currmatches[rolling_cols].fillna(currmatches[rolling_cols].mean())

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
currmatches['result'] = le.fit_transform(currmatches['result']) # W=2, D=0, L=1

In [ ]:
currmatches.info()